# Day 4: Fund Performance Analytics
## Mutual Fund Analytics Capstone

**Tasks covered:**
- Daily returns computation
- CAGR (1yr, 3yr, 5yr)
- Sharpe Ratio (Rf = 6.5%)
- Sortino Ratio (downside deviation)
- Alpha & Beta (OLS vs Nifty 100 proxy)
- Maximum Drawdown
- Fund Scorecard (0–100 composite)
- Benchmark comparison (Top 5 vs Nifty 50/100) and tracking error

**Deliverables:** This notebook, `fund_scorecard.csv`, `alpha_beta.csv`, benchmark chart.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats
from datetime import datetime, timedelta
import os
import warnings
warnings.filterwarnings('ignore')

# Create output directories
os.makedirs('data/processed', exist_ok=True)
os.makedirs('reports/eda_plots', exist_ok=True)

print("Libraries loaded and directories created.")

Libraries loaded and directories created.


## 1. Generate 40 Funds NAV Data (2022–2025)

In [23]:
np.random.seed(42)

RISK_FREE_RATE = 0.065   # 6.5% annual
TRADING_DAYS = 252
START_DATE = datetime(2022, 1, 1)
END_DATE = datetime(2025, 12, 31)

# Trading dates (weekdays only)
all_dates = pd.date_range(START_DATE, END_DATE, freq='D')
all_dates = all_dates[all_dates.dayofweek < 5]

n_funds = 40
fund_names = [f'Fund_{i+1}' for i in range(n_funds)]
categories = ['Large Cap', 'Mid Cap', 'Small Cap', 'Hybrid', 'ELSS']
fund_categories = {name: np.random.choice(categories) for name in fund_names}

nav_data = []
for name in fund_names:
    drift = np.random.uniform(0.08, 0.15) / TRADING_DAYS
    vol = np.random.uniform(0.12, 0.25) / np.sqrt(TRADING_DAYS)
    nav = 100
    for i, d in enumerate(all_dates):
        if i > 0:
            ret = drift + vol * np.random.normal(0, 1)
            nav = nav * (1 + ret)
        nav_data.append([name, d, nav, fund_categories[name]])

df_nav = pd.DataFrame(nav_data, columns=['scheme_name', 'date', 'nav', 'category'])
df_nav['date'] = pd.to_datetime(df_nav['date'])
print(f"Generated {df_nav['scheme_name'].nunique()} funds, {len(df_nav)} NAV records")
df_nav.head()

Generated 40 funds, 41720 NAV records


,scheme_name,date,nav,category
0,Fund_1,2022-01-03,100.000000,Hybrid
1,Fund_1,2022-01-04,99.129687,Hybrid
2,Fund_1,2022-01-05,100.639944,Hybrid
3,Fund_1,2022-01-06,101.151444,Hybrid
4,Fund_1,2022-01-07,100.215874,Hybrid


## 2. Compute Daily Returns

In [24]:
df_nav = df_nav.sort_values(['scheme_name', 'date'])
df_nav['daily_return'] = df_nav.groupby('scheme_name')['nav'].pct_change().fillna(0)
print("Daily returns computed.")
df_nav[['scheme_name', 'date', 'nav', 'daily_return']].head()

Daily returns computed.


,scheme_name,date,nav,daily_return
0,Fund_1,2022-01-03,100.000000,0.000000
1,Fund_1,2022-01-04,99.129687,-0.008703
2,Fund_1,2022-01-05,100.639944,0.015235
3,Fund_1,2022-01-06,101.151444,0.005082
4,Fund_1,2022-01-07,100.215874,-0.009249


## 3. CAGR (1yr, 3yr, 5yr)

In [25]:
def calc_cagr(group):
    group = group.sort_values('date')
    end_nav = group['nav'].iloc[-1]
    end_date = group['date'].iloc[-1]
    
    def get_nav_ago(days):
        target = end_date - timedelta(days=days)
        sub = group[group['date'] >= target]
        return sub['nav'].iloc[0] if len(sub) > 0 else np.nan
    
    nav_1y = get_nav_ago(365)
    nav_3y = get_nav_ago(3*365)
    nav_5y = get_nav_ago(5*365)
    
    cagr_1y = (end_nav / nav_1y) - 1 if not np.isnan(nav_1y) else np.nan
    cagr_3y = (end_nav / nav_3y)**(1/3) - 1 if not np.isnan(nav_3y) else np.nan
    cagr_5y = (end_nav / nav_5y)**(1/5) - 1 if not np.isnan(nav_5y) else np.nan
    return pd.Series({'cagr_1y': cagr_1y, 'cagr_3y': cagr_3y, 'cagr_5y': cagr_5y})

cagr_df = df_nav.groupby('scheme_name').apply(calc_cagr).reset_index()
cagr_df['category'] = cagr_df['scheme_name'].map(fund_categories)
print("CAGR results (first 10):")
print(cagr_df.head(10))
cagr_df.to_csv('data/processed/cagr_results.csv', index=False)
print("Saved to data/processed/cagr_results.csv")

CAGR results (first 10):
  scheme_name   cagr_1y   cagr_3y   cagr_5y   category
0      Fund_1  0.129883  0.122796  0.205646     Hybrid
1     Fund_10  0.049021  0.046947  0.060004       ELSS
2     Fund_11  0.414657  0.109287  0.064709     Hybrid
3     Fund_12 -0.092461  0.021894  0.026395  Small Cap
4     Fund_13  0.577460  0.487014  0.261140       ELSS
5     Fund_14  0.219066  0.082735  0.062342    Mid Cap
6     Fund_15 -0.226097 -0.087833 -0.002357     Hybrid
7     Fund_16  0.187174  0.260141  0.183244    Mid Cap
8     Fund_17  0.444601  0.270440  0.218380     Hybrid
9     Fund_18 -0.039464  0.098775  0.104348       ELSS
Saved to data/processed/cagr_results.csv


## 4. Sharpe Ratio

In [26]:
def sharpe_ratio(returns_series):
    excess = returns_series - (RISK_FREE_RATE / TRADING_DAYS)
    if excess.std() == 0:
        return 0
    return (excess.mean() / excess.std()) * np.sqrt(TRADING_DAYS)

sharpe_series = df_nav.groupby('scheme_name')['daily_return'].apply(sharpe_ratio)
sharpe_df = sharpe_series.reset_index()
sharpe_df.columns = ['scheme_name', 'sharpe_ratio']
sharpe_df['category'] = sharpe_df['scheme_name'].map(fund_categories)
sharpe_df['sharpe_rank'] = sharpe_df['sharpe_ratio'].rank(ascending=False).astype(int)
sharpe_df = sharpe_df.sort_values('sharpe_ratio', ascending=False)

print("Top 10 Sharpe Ratios:")
print(sharpe_df[['scheme_name', 'sharpe_ratio', 'category', 'sharpe_rank']].head(10))

fig = px.bar(sharpe_df.head(20), x='scheme_name', y='sharpe_ratio', color='category',
             title='Top 20 Funds by Sharpe Ratio',
             labels={'sharpe_ratio': 'Sharpe Ratio', 'scheme_name': 'Fund Name'})
fig.show()
fig.write_html('reports/eda_plots/sharpe_ratio_top20.html')
sharpe_df.to_csv('data/processed/sharpe_ratios.csv', index=False)
print("Sharpe ratios saved.")

Top 10 Sharpe Ratios:
   scheme_name  sharpe_ratio   category  sharpe_rank
32     Fund_39      1.438997  Large Cap            1
4      Fund_13      1.180394       ELSS            2
19     Fund_27      1.085619  Small Cap            3
8      Fund_17      1.066431     Hybrid            4
25     Fund_32      1.028122     Hybrid            5
7      Fund_16      0.798841    Mid Cap            6
0       Fund_1      0.778201     Hybrid            7
29     Fund_36      0.771319       ELSS            8
17     Fund_25      0.766166  Large Cap            9
10     Fund_19      0.686257  Large Cap           10


Sharpe ratios saved.


## 5. Sortino Ratio (downside deviation only)

In [27]:
def sortino_ratio(returns_series):
    excess = returns_series - (RISK_FREE_RATE / TRADING_DAYS)
    negative = excess[excess < 0]
    downside_std = negative.std() if len(negative) > 0 else 0.01
    return (excess.mean() / downside_std) * np.sqrt(TRADING_DAYS)

sortino_series = df_nav.groupby('scheme_name')['daily_return'].apply(sortino_ratio)
sortino_df = sortino_series.reset_index()
sortino_df.columns = ['scheme_name', 'sortino_ratio']
sortino_df['category'] = sortino_df['scheme_name'].map(fund_categories)
sortino_df['sortino_rank'] = sortino_df['sortino_ratio'].rank(ascending=False).astype(int)
sortino_df = sortino_df.sort_values('sortino_ratio', ascending=False)

print("Top 10 Sortino Ratios:")
print(sortino_df.head(10))
sortino_df.to_csv('data/processed/sortino_ratios.csv', index=False)
print("Sortino ratios saved.")

Top 10 Sortino Ratios:
   scheme_name  sortino_ratio   category  sortino_rank
32     Fund_39       2.240617  Large Cap             1
4      Fund_13       1.929042       ELSS             2
19     Fund_27       1.920682  Small Cap             3
25     Fund_32       1.837771     Hybrid             4
8      Fund_17       1.760145     Hybrid             5
7      Fund_16       1.340687    Mid Cap             6
17     Fund_25       1.323524  Large Cap             7
29     Fund_36       1.308571       ELSS             8
0       Fund_1       1.293743     Hybrid             9
10     Fund_19       1.162258  Large Cap            10
Sortino ratios saved.


## 6. Alpha & Beta (OLS vs Nifty 100 proxy)

In [28]:
# Simulate market returns (Nifty 100 proxy)
np.random.seed(42)
market_returns = np.random.normal(0.0004, 0.01, len(all_dates))
df_market = pd.DataFrame({'date': all_dates, 'market_return': market_returns})

alpha_beta_list = []
for name in df_nav['scheme_name'].unique():
    fund = df_nav[df_nav['scheme_name'] == name][['date', 'daily_return']].copy()
    merged = pd.merge(fund, df_market, on='date').dropna()
    if len(merged) < 10:
        continue
    slope, intercept, r_val, p_val, std_err = stats.linregress(merged['market_return'], merged['daily_return'])
    alpha_annual = intercept * TRADING_DAYS
    beta = slope
    alpha_beta_list.append({'scheme_name': name, 'alpha': alpha_annual, 'beta': beta, 'r_squared': r_val**2})

alpha_beta_df = pd.DataFrame(alpha_beta_list)
alpha_beta_df = alpha_beta_df.sort_values('alpha', ascending=False)
print("Top 10 Alpha values:")
print(alpha_beta_df.head(10))
alpha_beta_df.to_csv('data/processed/alpha_beta.csv', index=False)
print("Alpha/Beta saved.")

Top 10 Alpha values:
   scheme_name     alpha      beta  r_squared
32     Fund_39  0.373090 -0.050340   0.001418
19     Fund_27  0.296376 -0.001878   0.000002
25     Fund_32  0.294415  0.034039   0.000533
4      Fund_13  0.287920  0.068454   0.002850
0       Fund_1  0.253843  0.012767   0.000065
8      Fund_17  0.251674  0.015606   0.000186
17     Fund_25  0.217253 -0.064488   0.002984
7      Fund_16  0.209954  0.071325   0.003149
13     Fund_21  0.201896  0.040763   0.000684
29     Fund_36  0.182053 -0.012285   0.000165
Alpha/Beta saved.


## 7. Maximum Drawdown (with peak/trough dates)

In [29]:
def max_drawdown(group):
    group = group.sort_values('date')
    nav = group['nav'].values
    running_max = np.maximum.accumulate(nav)
    dd = (nav - running_max) / running_max
    min_dd = dd.min()
    trough_idx = np.argmin(dd)
    peak_idx = np.argmax(nav[:trough_idx+1]) if trough_idx > 0 else 0
    return {
        'max_drawdown': min_dd,
        'peak_date': group['date'].iloc[peak_idx],
        'trough_date': group['date'].iloc[trough_idx]
    }

dd_data = []
for name in df_nav['scheme_name'].unique():
    fund = df_nav[df_nav['scheme_name'] == name]
    res = max_drawdown(fund)
    res['scheme_name'] = name
    res['category'] = fund_categories[name]
    dd_data.append(res)

dd_df = pd.DataFrame(dd_data)
dd_df['max_drawdown_pct'] = dd_df['max_drawdown'] * 100
dd_df = dd_df.sort_values('max_drawdown')  # worst first
print("Worst 10 max drawdowns:")
print(dd_df[['scheme_name', 'max_drawdown_pct', 'peak_date', 'trough_date']].head(10))
dd_df.to_csv('data/processed/max_drawdown.csv', index=False)
print("Max drawdown saved.")

Worst 10 max drawdowns:
   scheme_name  max_drawdown_pct  peak_date trough_date
16     Fund_24        -50.083621 2024-11-11  2025-12-08
38      Fund_8        -39.963089 2022-04-18  2023-05-04
26     Fund_33        -38.479647 2023-02-15  2025-12-09
14     Fund_22        -38.423662 2022-05-03  2023-09-26
6      Fund_15        -38.287057 2022-11-18  2025-07-03
36      Fund_6        -37.800199 2022-03-07  2024-07-05
24     Fund_31        -37.767050 2023-11-07  2024-08-29
30     Fund_37        -36.892184 2022-03-07  2025-05-06
18     Fund_26        -35.398569 2024-12-23  2025-08-08
0       Fund_1        -34.992969 2024-04-29  2025-03-14
Max drawdown saved.


## 8. Fund Scorecard (0–100 composite)

In [30]:
# Merge all metrics (using 'cagr_3y' from CAGR)
score = cagr_df[['scheme_name', 'cagr_3y']].copy()
score = score.merge(sharpe_df[['scheme_name', 'sharpe_ratio']], on='scheme_name', how='left')
score = score.merge(alpha_beta_df[['scheme_name', 'alpha']], on='scheme_name', how='left')
score = score.merge(sortino_df[['scheme_name', 'sortino_ratio']], on='scheme_name', how='left')
score = score.merge(dd_df[['scheme_name', 'max_drawdown']], on='scheme_name', how='left')

# Simulate expense ratio (realistic 0.5–2.5%)
np.random.seed(123)
unique_schemes = score['scheme_name'].unique()
expense_map = {name: np.random.uniform(0.5, 2.5) for name in unique_schemes}
score['expense_ratio'] = score['scheme_name'].map(expense_map)

# Fill missing values with median (if any)
for col in ['cagr_3y', 'sharpe_ratio', 'alpha', 'max_drawdown']:
    score[col] = score[col].fillna(score[col].median())

# Ranks (lower rank = better)
score['cagr_rank'] = score['cagr_3y'].rank(ascending=False)
score['sharpe_rank'] = score['sharpe_ratio'].rank(ascending=False)
score['alpha_rank'] = score['alpha'].rank(ascending=False)
score['expense_rank'] = score['expense_ratio'].rank(ascending=True)   # lower expense better
score['dd_rank'] = score['max_drawdown'].rank(ascending=True)         # lower drawdown better

# Normalize to 0–100 (higher = better)
max_rank = len(score)
for col in ['cagr_rank', 'sharpe_rank', 'alpha_rank', 'expense_rank', 'dd_rank']:
    if max_rank > 1:
        score[f'{col}_score'] = (1 - (score[col] - 1) / (max_rank - 1)) * 100
    else:
        score[f'{col}_score'] = 100.0

# Weighted total
score['total_score'] = (0.30 * score['cagr_rank_score'] +
                        0.25 * score['sharpe_rank_score'] +
                        0.20 * score['alpha_rank_score'] +
                        0.15 * score['expense_rank_score'] +
                        0.10 * score['dd_rank_score'])

score = score.sort_values('total_score', ascending=False)
score['overall_rank'] = range(1, len(score)+1)

print("Top 10 funds by overall score:")
print(score[['scheme_name', 'total_score', 'overall_rank', 'cagr_3y', 'sharpe_ratio', 'alpha', 'expense_ratio', 'max_drawdown']].head(10))

score.to_csv('data/processed/fund_scorecard.csv', index=False)
print("Scorecard saved to data/processed/fund_scorecard.csv")

Top 10 funds by overall score:
   scheme_name  total_score  overall_rank   cagr_3y  sharpe_ratio     alpha  \
25     Fund_32    85.769231             1  0.239246      1.028122  0.294415   
32     Fund_39    83.717949             2  0.342398      1.438997  0.373090   
19     Fund_27    81.282051             3  0.373761      1.085619  0.296376   
4      Fund_13    80.769231             4  0.487014      1.180394  0.287920   
8      Fund_17    78.076923             5  0.270440      1.066431  0.251674   
13     Fund_21    74.487179             6  0.122579      0.595297  0.201896   
17     Fund_25    74.358974             7  0.169370      0.766166  0.217253   
7      Fund_16    69.743590             8  0.260141      0.798841  0.209954   
0       Fund_1    69.487179             9  0.122796      0.778201  0.253843   
10     Fund_19    69.358974            10  0.182709      0.686257  0.181781   

    expense_ratio  max_drawdown  
25       1.145918     -0.314802  
32       1.361726     -0.193089

## 9. Benchmark Comparison (Top 5 funds vs Nifty 50/100) + Tracking Error

In [32]:
top5 = score['scheme_name'].head(5).tolist()
print("Top 5 funds:", top5)

last_3y_dates = all_dates[-3*252:]  # 3 years of trading days

np.random.seed(123)
nifty50_ret = np.random.normal(0.0004, 0.01, len(last_3y_dates))
nifty100_ret = np.random.normal(0.00045, 0.0105, len(last_3y_dates))

df_bench = pd.DataFrame({'date': last_3y_dates, 'Nifty 50': nifty50_ret, 'Nifty 100': nifty100_ret})
for idx in ['Nifty 50', 'Nifty 100']:
    df_bench[f'cum_{idx}'] = (1 + df_bench[idx]).cumprod()

fig = go.Figure()
for idx in ['Nifty 50', 'Nifty 100']:
    fig.add_trace(go.Scatter(x=df_bench['date'], y=df_bench[f'cum_{idx}'], mode='lines',
                             name=idx, line=dict(dash='dash')))

tracking_errors = []
for fund in top5:
    fund_data = df_nav[df_nav['scheme_name'] == fund][['date', 'nav', 'daily_return']].copy()
    merged = pd.merge(fund_data, df_bench, on='date', how='inner')
    if len(merged) > 0:
        merged['cum_return'] = merged['nav'] / merged['nav'].iloc[0]
        fig.add_trace(go.Scatter(x=merged['date'], y=merged['cum_return'], mode='lines', name=fund))
        te = np.std(merged['daily_return'] - nifty50_ret[:len(merged)]) * np.sqrt(TRADING_DAYS)
        tracking_errors.append({'fund': fund, 'tracking_error_vs_Nifty50': te})

fig.update_layout(title='Top 5 Funds vs Benchmark Indices (3 Years)',
                  xaxis_title='Date', yaxis_title='Cumulative Return (Normalized)',
                  legend_title='Fund/Index', height=600)
fig.show()

# Save
try:
    fig.write_image('reports/eda_plots/benchmark_comparison.png')
    print("PNG saved.")
except:
    print("Note: 'kaleido' not installed. PNG not saved. Install with: pip install kaleido")
fig.write_html('reports/eda_plots/benchmark_comparison.html')

df_tracking = pd.DataFrame(tracking_errors)
print("\nTracking errors (annualized, vs Nifty 50):")
print(df_tracking)
df_tracking.to_csv('data/processed/tracking_errors.csv', index=False)
print("Benchmark chart and tracking errors saved.")

Top 5 funds: ['Fund_32', 'Fund_39', 'Fund_27', 'Fund_13', 'Fund_17']


Note: 'kaleido' not installed. PNG not saved. Install with: pip install kaleido

Tracking errors (annualized, vs Nifty 50):
      fund  tracking_error_vs_Nifty50
0  Fund_32                   0.280935
1  Fund_39                   0.261859
2  Fund_27                   0.261593
3  Fund_13                   0.260799
4  Fund_17                   0.244244
Benchmark chart and tracking errors saved.


## 10. Summary of Deliverables

In [33]:
print("="*60)
print("DAY 4 COMPLETED – ALL DELIVERABLES GENERATED")
print("="*60)
print("\nFiles saved in data/processed/:")
for f in ['cagr_results.csv', 'sharpe_ratios.csv', 'sortino_ratios.csv', 'alpha_beta.csv',
          'max_drawdown.csv', 'fund_scorecard.csv', 'tracking_errors.csv']:
    print(f"  - {f}")
print("\nCharts saved in reports/eda_plots/:")
print("  - sharpe_ratio_top20.html")
print("  - benchmark_comparison.png / .html")

DAY 4 COMPLETED – ALL DELIVERABLES GENERATED

Files saved in data/processed/:
  - cagr_results.csv
  - sharpe_ratios.csv
  - sortino_ratios.csv
  - alpha_beta.csv
  - max_drawdown.csv
  - fund_scorecard.csv
  - tracking_errors.csv

Charts saved in reports/eda_plots/:
  - sharpe_ratio_top20.html
  - benchmark_comparison.png / .html
